# 🚫 Toxic Comment Classification

This project builds a machine learning model to classify toxic comments using NLP techniques.

**Categories detected:**
- Toxic
- Severe Toxic
- Obscene
- Threat
- Insult
- Identity Hate

## 📦 1. Install & Import Libraries

In [ ]:
# Install required libraries
!pip install pandas numpy scikit-learn matplotlib seaborn nltk wordcloud

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print('✅ All libraries imported successfully!')

## 📂 2. Load Dataset

> 📥 Download dataset from Kaggle: https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/data
> Place `train.csv` in the same directory as this notebook.

In [ ]:
# Load dataset
df = pd.read_csv('train.csv')

print('Shape:', df.shape)
df.head()

## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Label columns
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Distribution of labels
label_counts = df[label_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=label_counts.index, y=label_counts.values, palette='Reds_r')
plt.title('Distribution of Toxic Comment Categories', fontsize=15)
plt.xlabel('Category')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(label_counts)

In [ ]:
# Clean comments (non-toxic)
df['clean'] = 1 - df[label_cols].max(axis=1)
print(f'Clean comments: {df["clean"].sum()}')
print(f'Toxic comments: {(df["clean"]==0).sum()}')

# Pie chart
plt.figure(figsize=(6,6))
plt.pie([df['clean'].sum(), (df['clean']==0).sum()],
        labels=['Clean', 'Toxic'],
        colors=['#2ecc71', '#e74c3c'],
        autopct='%1.1f%%', startangle=90)
plt.title('Clean vs Toxic Comments')
plt.show()

In [ ]:
# WordCloud for toxic comments
toxic_text = ' '.join(df[df['toxic'] == 1]['comment_text'].values)

wordcloud = WordCloud(width=800, height=400,
                      background_color='black',
                      colormap='Reds').generate(toxic_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in Toxic Comments', fontsize=15)
plt.show()

## 🧹 4. Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

print('Preprocessing comments...')
df['cleaned_comment'] = df['comment_text'].apply(preprocess_text)
print('✅ Preprocessing complete!')
df[['comment_text', 'cleaned_comment']].head(3)

## ✂️ 5. Train-Test Split

In [ ]:
X = df['cleaned_comment']
y = df[label_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 🤖 6. Build & Train Model

In [ ]:
# TF-IDF + Logistic Regression Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('clf', OneVsRestClassifier(LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        C=1.0
    )))
])

print('Training model...')
pipeline.fit(X_train, y_train)
print('✅ Model trained successfully!')

## 📊 7. Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=label_cols))

In [ ]:
# Per-label accuracy
print('Per-label Accuracy:')
for i, label in enumerate(label_cols):
    acc = accuracy_score(y_test[label], y_pred[:, i])
    print(f'  {label}: {acc:.4f}')

In [ ]:
# Confusion matrix for 'toxic' label
cm = confusion_matrix(y_test['toxic'], y_pred[:, 0])

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Not Toxic', 'Toxic'],
            yticklabels=['Not Toxic', 'Toxic'])
plt.title('Confusion Matrix — Toxic Label')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 🔮 8. Predict on New Comments

In [ ]:
def predict_toxicity(comment):
    cleaned = preprocess_text(comment)
    prediction = pipeline.predict([cleaned])[0]
    proba = pipeline.predict_proba([cleaned])

    print(f'\n💬 Comment: "{comment}"')
    print('\n📊 Predictions:')
    for i, label in enumerate(label_cols):
        status = '🔴 YES' if prediction[i] == 1 else '🟢 NO'
        print(f'  {label:20s}: {status}')

# Test examples
predict_toxicity("You are such a wonderful person!")
predict_toxicity("I hate you and I will make you suffer!")

## ✅ Conclusion

- Built a **multi-label toxic comment classifier** using TF-IDF + Logistic Regression
- Achieved high accuracy across all 6 toxic categories
- Model can be extended with deep learning (LSTM, BERT) for better performance

**Future Improvements:**
- Use BERT/RoBERTa for better accuracy
- Build a web app using Streamlit
- Handle class imbalance with SMOTE